In [1]:
"""
DQN (Deep Q-Network) - CartPole 示例
=====================================
论文: Playing Atari with Deep Reinforcement Learning (DeepMind, 2013)

核心技术:
  1. Experience Replay     - 打破样本相关性
  2. Target Network        - 稳定 Q 值目标
  3. Epsilon-Greedy        - 探索与利用的平衡

环境: CartPole-v1
  状态: [车位置, 车速度, 杆角度, 杆角速度] (4维)
  动作: {0: 向左推, 1: 向右推}
  目标: 保持杆子不倒，单回合最大 500 步
"""

import random
import numpy as np
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim

import gymnasium as gym


# ─────────────────────────────────────────
# 1. Q 网络 (Neural Network)
# ─────────────────────────────────────────
class QNetwork(nn.Module):
    """
    输入: 状态向量 s
    输出: 每个动作的 Q 值 Q(s, a)
    """
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ─────────────────────────────────────────
# 2. 经验回放缓冲区 (Replay Buffer)
# ─────────────────────────────────────────
class ReplayBuffer:
    """
    存储 (s, a, r, s', done) 元组
    随机采样打破时序相关性，提升训练稳定性
    """
    def __init__(self, capacity: int = 10_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)


# ─────────────────────────────────────────
# 3. DQN Agent
# ─────────────────────────────────────────
class DQNAgent:
    def __init__(
        self,
        state_dim: int,
        action_dim: int,
        lr: float = 1e-3,
        gamma: float = 0.99,          # 折扣因子
        epsilon_start: float = 1.0,   # 初始探索率
        epsilon_end: float = 0.05,    # 最小探索率
        epsilon_decay: float = 0.995, # 衰减系数
        buffer_size: int = 10_000,
        batch_size: int = 64,
        target_update_freq: int = 10, # 每 N 回合同步 Target Network
    ):
        self.action_dim = action_dim
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.train_step = 0

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Online Network: 实时更新
        self.q_net = QNetwork(state_dim, action_dim).to(self.device)
        # Target Network: 定期同步，用于生成稳定的 TD 目标
        self.target_net = QNetwork(state_dim, action_dim).to(self.device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.replay_buffer = ReplayBuffer(buffer_size)

    # ── 动作选择 (ε-greedy) ──────────────────
    def select_action(self, state: np.ndarray) -> int:
        """
        以 epsilon 概率随机探索，否则贪心选择最优动作
        """
        if random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q_values = self.q_net(state_t)
        return q_values.argmax(dim=1).item()

    # ── 存储经验 ─────────────────────────────
    def store(self, state, action, reward, next_state, done):
        self.replay_buffer.push(state, action, reward, next_state, done)

    # ── 训练一步 ─────────────────────────────
    def train(self):
        if len(self.replay_buffer) < self.batch_size:
            return None

        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)

        states_t      = torch.FloatTensor(states).to(self.device)
        actions_t     = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        rewards_t     = torch.FloatTensor(rewards).unsqueeze(1).to(self.device)
        next_states_t = torch.FloatTensor(next_states).to(self.device)
        dones_t       = torch.FloatTensor(dones).unsqueeze(1).to(self.device)

        # 当前 Q 值: Q(s, a)
        current_q = self.q_net(states_t).gather(1, actions_t)

        # TD 目标: r + γ * max_a' Q_target(s', a')
        with torch.no_grad():
            max_next_q = self.target_net(next_states_t).max(dim=1, keepdim=True)[0]
            target_q = rewards_t + self.gamma * max_next_q * (1 - dones_t)

        # Huber Loss (比 MSE 对异常值更鲁棒)
        loss = nn.SmoothL1Loss()(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        # 梯度裁剪，防止梯度爆炸
        nn.utils.clip_grad_norm_(self.q_net.parameters(), max_norm=10)
        self.optimizer.step()

        self.train_step += 1
        return loss.item()

    # ── 同步 Target Network ───────────────────
    def sync_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())

    # ── 衰减 epsilon ──────────────────────────
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)


# ─────────────────────────────────────────
# 4. 训练主循环
# ─────────────────────────────────────────
def train_dqn(num_episodes: int = 400, render: bool = False):
    env = gym.make("CartPole-v1", render_mode="human" if render else None)
    state_dim  = env.observation_space.shape[0]   # 4
    action_dim = env.action_space.n               # 2

    agent = DQNAgent(state_dim, action_dim)

    reward_history = []
    best_avg = -float("inf")

    print(f"设备: {agent.device}")
    print(f"状态维度: {state_dim}, 动作维度: {action_dim}")
    print("=" * 55)
    print(f"{'Episode':>8} | {'Reward':>8} | {'Avg(50)':>8} | {'Epsilon':>8}")
    print("=" * 55)

    for episode in range(1, num_episodes + 1):
        state, _ = env.reset()
        total_reward = 0
        done = False

        while not done:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # 提前结束给负奖励，加速学习
            shaped_reward = reward if not terminated else -10.0

            agent.store(state, action, shaped_reward, next_state, done)
            agent.train()

            state = next_state
            total_reward += reward

        agent.decay_epsilon()

        # 定期同步 Target Network
        if episode % agent.target_update_freq == 0:
            agent.sync_target()

        reward_history.append(total_reward)
        avg50 = np.mean(reward_history[-50:])

        if episode % 20 == 0:
            status = " ← best!" if avg50 > best_avg else ""
            print(f"{episode:>8} | {total_reward:>8.1f} | {avg50:>8.1f} | {agent.epsilon:>8.3f}{status}")
            if avg50 > best_avg:
                best_avg = avg50
                torch.save(agent.q_net.state_dict(), "dqn_best.pth")

        # 连续 50 回合平均 ≥ 475 视为解决
        if avg50 >= 475 and episode >= 50:
            print(f"\n✅ 问题解决！Episode {episode}, 最近50回合平均奖励: {avg50:.1f}")
            break

    env.close()
    print(f"\n训练完成。最佳平均奖励: {best_avg:.1f}")
    return agent, reward_history


# ─────────────────────────────────────────
# 5. 评估已训练的模型
# ─────────────────────────────────────────
def evaluate(agent: DQNAgent, num_episodes: int = 10, render: bool = True):
    env = gym.make("CartPole-v1", render_mode="human" if render else None)
    agent.epsilon = 0.0  # 关闭探索，纯贪心

    rewards = []
    for ep in range(num_episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False
        while not done:
            action = agent.select_action(state)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
        rewards.append(total_reward)
        print(f"评估 Episode {ep+1}: {total_reward:.0f} 步")

    env.close()
    print(f"\n平均得分: {np.mean(rewards):.1f} ± {np.std(rewards):.1f}")


# ─────────────────────────────────────────
# 入口
# ─────────────────────────────────────────
if __name__ == "__main__":
    agent, history = train_dqn(num_episodes=400)
    evaluate(agent, render=True)  # 取消注释可视化评估

设备: cpu
状态维度: 4, 动作维度: 2
 Episode |   Reward |  Avg(50) |  Epsilon
      20 |     19.0 |     24.6 |    0.905 ← best!
      40 |     55.0 |     24.6 |    0.818
      60 |     10.0 |     28.1 |    0.740 ← best!
      80 |     26.0 |     34.6 |    0.670 ← best!
     100 |     14.0 |     35.6 |    0.606 ← best!
     120 |     36.0 |     40.9 |    0.548 ← best!
     140 |     27.0 |     44.4 |    0.496 ← best!
     160 |     20.0 |     46.4 |    0.448 ← best!
     180 |    117.0 |     45.2 |    0.406
     200 |    113.0 |     43.4 |    0.367
     220 |     12.0 |     49.0 |    0.332 ← best!
     240 |     26.0 |     68.8 |    0.300 ← best!
     260 |     84.0 |     89.8 |    0.272 ← best!
     280 |    148.0 |    100.5 |    0.246 ← best!
     300 |    162.0 |    102.3 |    0.222 ← best!
     320 |    437.0 |    142.0 |    0.201 ← best!
     340 |    123.0 |    221.6 |    0.182 ← best!
     360 |    212.0 |    227.0 |    0.165 ← best!
     380 |    179.0 |    195.9 |    0.149
     400 |    2

KeyboardInterrupt: 